In [1]:
#pip installs
#%pip install langchain
#%pip install langchain-ollama
#%pip install langchain-qdrant
#%pip install langchain-qdrant -U
#%pip install pandas
#%pip install fastembed

In [2]:
# import the main libraries
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import Distance, VectorParams
import os
import pandas as pd

/Users/jen/Documents/CSC174_AdvDatabases/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Use Qdrant to create a memory collection and vector store
- collection is named "movie_collection" 
- use COSINE distance

In [3]:
qdrant_client = QdrantClient(":memory:")
embeddings = OllamaEmbeddings(model="mxbai-embed-large")

# create the collection in Qdrant
qdrant_client.create_collection(
    collection_name="movie_collection",
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE)
)

# create the vector store 
vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name="movie_collection", # must be the same as the collection created in Qdrant
    embedding=embeddings
)


# 2. Process “Public Criterion Master List.csv”
- include each field in the metadata, as well as “source: csv”
- “documents” part can include any fields, must at least have title and description

In [4]:
# process the csv
df = pd.read_csv("./dataset/Public Criterion Master List.csv")
csv_list = df.to_dict(orient="records")

csv_docs = [Document(
        page_content="Spine:"+ (str(doc["SPINE"]))+ " " + "Title:"+(str(doc["TITLE (Year)"]))+ \
                    "Director:"+(str(doc["DIRECTOR"]))+ " " + "Country:"+(str(doc["COUNTRY"]))+ " " + \
                    "Description:"+(str(doc["Brief Description"])), 
        metadata={
            "source": "csv", 
            "spine": str(doc["SPINE"]),
            "title_year": str(doc["TITLE (Year)"]),
            "director": str(doc["DIRECTOR"]),
            "country": str(doc["COUNTRY"]),
            "4k_release": str(doc["4K Release?"]),
            "imdb_rating": str(doc["IMDb Rating"]),
            "rotten_tomatoes": str(doc["Rotten Tomatoes"]),
            "count_sort": str(doc["Count (Sort)"]),
            "year": str(doc["YEAR"]),
            "brief_description": str(doc["Brief Description"]),
        }) for doc in csv_list]


# 3. Add the processed data to the vector store with add_documents()

In [5]:
ids = [i for i in range(len(csv_list))]
vector_store.add_documents(
    documents=csv_docs,
    ids=ids
)

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76]

# 4. Retrieve the point with id 1, using get_by_ids()

In [6]:
vector_store.get_by_ids([1])

[Document(metadata={'source': 'csv', 'spine': '2', 'title_year': 'Seven Samurai (1954)', 'director': 'Akira Kurosawa', 'country': 'Japan', '4k_release': 'True', 'imdb_rating': '8.6/10', 'rotten_tomatoes': '100%', 'count_sort': '2', 'year': '1954.0', 'brief_description': 'A poor village hires seven desperate but experienced samurai to protect them from a ruthless band of marauding bandits.', '_id': 1, '_collection_name': 'movie_collection'}, page_content='Spine:2 Title:Seven Samurai (1954)Director:Akira Kurosawa Country:Japan Description:A poor village hires seven desperate but experienced samurai to protect them from a ruthless band of marauding bandits.')]

# 5. Upsert (add.documents()) "dummy data" to id 1
- use get_by_ids to check

In [7]:
# create a dummy document
dummy_doc = Document(
    page_content="dummy data",
    metadata={"source": "dummy"}
)
# add to the vector store
vector_store.add_documents(documents=[dummy_doc], ids=[1])
# check the added document
vector_store.get_by_ids([1])


[Document(metadata={'source': 'dummy', '_id': 1, '_collection_name': 'movie_collection'}, page_content='dummy data')]

# 6. Delete point id 1

In [8]:
vector_store.delete(ids=[1])

True

# 7. Use add_documents() to restore the original data for id 1

In [9]:
vector_store.add_documents(documents=[csv_docs[1]], ids=[1])

[1]

# 8. Use get_by_ids to check if it has been inserted successfully

In [10]:
vector_store.get_by_ids([1])

[Document(metadata={'source': 'csv', 'spine': '2', 'title_year': 'Seven Samurai (1954)', 'director': 'Akira Kurosawa', 'country': 'Japan', '4k_release': 'True', 'imdb_rating': '8.6/10', 'rotten_tomatoes': '100%', 'count_sort': '2', 'year': '1954.0', 'brief_description': 'A poor village hires seven desperate but experienced samurai to protect them from a ruthless band of marauding bandits.', '_id': 1, '_collection_name': 'movie_collection'}, page_content='Spine:2 Title:Seven Samurai (1954)Director:Akira Kurosawa Country:Japan Description:A poor village hires seven desperate but experienced samurai to protect them from a ruthless band of marauding bandits.')]

# 9. Construct a query statement to get 10 results using similarity_search_with_relevance_scores 
- “Which of these movies deal with war in some way?” Print the results with relevance scores.
- Then, use a score threshold to narrow down the results and print them again

In [11]:

q1="""Which of these movies deal with war in some way?"""
query_results1 = vector_store.similarity_search_with_relevance_scores(query=q1, k=10)
for doc, score in query_results1:
    print(f"Score: {score:.4f}\nContent: {doc.page_content}\nMetadata: {doc.metadata}\n")

Score: 0.7856
Content: Spine:17 Title:Salò, or The 120 Days of Sodom (1976)Director:Pier Paolo Pasolini Country:Italy Description:A controversial and explicit film set during WWII, where four wealthy men abduct teenagers and subject them to extreme torture and sexual degradation.
Metadata: {'source': 'csv', 'spine': '17', 'title_year': 'Salò, or The 120 Days of Sodom (1976)', 'director': 'Pier Paolo Pasolini', 'country': 'Italy', '4k_release': 'False', 'imdb_rating': '6.8/10', 'rotten_tomatoes': '69%', 'count_sort': '17', 'year': '1976.0', 'brief_description': 'A controversial and explicit film set during WWII, where four wealthy men abduct teenagers and subject them to extreme torture and sexual degradation.', '_id': 16, '_collection_name': 'movie_collection'}

Score: 0.7854
Content: Spine:2 Title:Seven Samurai (1954)Director:Akira Kurosawa Country:Japan Description:A poor village hires seven desperate but experienced samurai to protect them from a ruthless band of marauding bandits.


In [12]:
# print again with a threshold 
threshold = 0.7800
query_results1 = vector_store.similarity_search_with_relevance_scores(query=q1, k=10, score_threshold=threshold)
for doc, score in query_results1:
    print(f"Score: {score:.4f}\nContent: {doc.page_content}\nMetadata: {doc.metadata}\n")

Score: 0.7856
Content: Spine:17 Title:Salò, or The 120 Days of Sodom (1976)Director:Pier Paolo Pasolini Country:Italy Description:A controversial and explicit film set during WWII, where four wealthy men abduct teenagers and subject them to extreme torture and sexual degradation.
Metadata: {'source': 'csv', 'spine': '17', 'title_year': 'Salò, or The 120 Days of Sodom (1976)', 'director': 'Pier Paolo Pasolini', 'country': 'Italy', '4k_release': 'False', 'imdb_rating': '6.8/10', 'rotten_tomatoes': '69%', 'count_sort': '17', 'year': '1976.0', 'brief_description': 'A controversial and explicit film set during WWII, where four wealthy men abduct teenagers and subject them to extreme torture and sexual degradation.', '_id': 16, '_collection_name': 'movie_collection'}

Score: 0.7854
Content: Spine:2 Title:Seven Samurai (1954)Director:Akira Kurosawa Country:Japan Description:A poor village hires seven desperate but experienced samurai to protect them from a ruthless band of marauding bandits.


# 10. Process “Five Movies.pdf”
- use Langchain pdf loader and text splitter

In [13]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

filepath="./dataset/Five Movies.pdf"
loader = PyPDFLoader(filepath)
document=loader.load()

c_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " "],
    chunk_size=800,
    chunk_overlap=10
)

documents = c_splitter.split_documents(document)

/var/folders/yj/n_z_slv1029_2t6ck3q5hvn80000gn/T/ipykernel_81186/2869942781.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


# 11. Add the processed pdf files to the same vector store collection
- start at an available id index, not 0

In [14]:
# grab starting index for new entries
response = qdrant_client.count(
    collection_name="movie_collection",
    exact=True
)
# add the new entries from the startIdx to the vector_store
startIdx = response.count
custom_ids = [startIdx + i for i in range(len(documents))]
vector_store.add_documents(documents=documents, ids=custom_ids)


[77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88]

# 12. Use Langchain TextLoader to process “Twelve Angry Men Script.txt" 
- use encoding='utf-8'

In [15]:
from langchain_community.document_loaders import TextLoader


loader = TextLoader("./dataset/Twelve Angry Men Script.txt", encoding="utf-8")
txt_docs = loader.load()
t_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " "],
    chunk_size=800,
    chunk_overlap=100
)
chunks = t_splitter.split_documents(txt_docs)

# Load the processed text document into the vector store

In [16]:
# grab starting index for new entries
response = qdrant_client.count(
    collection_name="movie_collection",
    exact=True
)
# add the new entries from the startIdx to the vector_store
idx = response.count
ids = [idx + i for i in range(len(chunks))]
vector_store.add_documents(
    documents=chunks,
    ids=ids
)

[89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182]

# 14. Query again with "What can you tell me about Twelve Angry Men? Please include some specific lines." 
- Print results with relevance scores 

In [17]:
q2="""What can you tell me about Twelve Angry Men? Please include some specific lines."""
query_results2 = vector_store.similarity_search_with_relevance_scores(query=q2, k=10)
for doc, score in query_results2:
    print(f"Score: {score:.4f}\nContent: {doc.page_content}\nMetadata: {doc.metadata}\n")

Score: 0.8730
Content: Twelve Angry Men

by REGINALD ROSE 

The following play was written for television. It begins with a list of characters, but on television you would not have the 
benefit of that list (although, of course, you would be able to tell something about each character from his appearance on the 
screen). Imagine, then, on the screen of your mind the scene described at the start: a courtroom, with the area, or box, in the 
foreground for the twelve members of the jury. The kinds of men they are will become clear as the play unfolds. One 
suggestion, though: pay particular attention to juror NO. 8: And it might be helpful to fix your mind the equation: 7+3=10. 
For jurors 7, 3, and 10 also play extremely important parts.  

CHARACTERS
Metadata: {'source': './dataset/Twelve Angry Men Script.txt', '_id': 89, '_collection_name': 'movie_collection'}

Score: 0.8622
Content: Spine:591 Title:12 Angry Men (1957)Director:Sidney Lumet Country:United States Description:A jury of 12

# 15. Use LCEL to construct a RAG chain

In [18]:
# create a retriever for the vector store
retriever = vector_store.as_retriever(
    search_kwargs={"k": 10} 
) 
# function to format documents
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama.llms import OllamaLLM

model = OllamaLLM(model="llama3.2:3b", temperature=5)

template = """
You are a helpful assistant. 
Your users are asking questions about information contained in the dataset folder.
If you do not know the answer, please say I do not know.

Here are some relevant information: {information}

Here is the question to answer: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [20]:
#chain = prompt | model
chain = (
    {"information": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

# 16. Query with the question from before and add two more queries

In [21]:
question="What can you tell me about Twelve Angry Men? Please include some specific lines."
response = chain.invoke(question)
print(response)

Twelve Angry Men is a 1957 drama film directed by Sidney Lumet, based on the 1954 television teleplay of the same name by Reginald Rose. The film takes place in a jury room where 12 men must deliberate the case of a young man accused of murdering his father.

In the film, Juror #8 (played by Henry Fonda) is a dissenting member who challenges the convictions of his fellow jurors, including Juror #3 (played by Edward G. Robinson) who believes that the accused is guilty. As the deliberation continues, several jurors reveal their personal backgrounds and biases, including Juror #9 (played by Jack Warden) who has an honest look at NO. 9's words "NO. 9: It's only one night. A man may die.," and Juror #10 (played by Henry Rearden) who admits that most of his friends are unemotional.

Juror #8's line, "There's no good or bad in the individual, just good or bad in the situation," highlights the importance of considering all the evidence in a trial. He is later shown as standing for what he beli

In [22]:
# additional question 1
question="What themes does 12 Angry Men share with other movies in the dataset? List the titles of similar movies"
response = chain.invoke(question)
print(response)

Based on the provided information, 12 Angry Men shares themes related to:

1. The American justice system and its limitations.
2. Racial tensions and prejudice.
3. Moral dilemma and the importance of individual responsibility.
4. Social change and the evolution of societal attitudes.

Similar movies that may explore these themes include:

1. To Kill a Mockingbird (1962) - Also by Sidney Lumet, exploring racial tensions and injustice in a courtroom setting.
2. 12 Years a Slave (2013) - A historical drama that deals with themes of injustice, morality, and the American justice system.
3. The Shawshank Redemption (1994) - A drama about hope, redemption, and the struggle for individual freedom, set against the backdrop of the American prison system.
4. Miss Daisy (1989) - A drama that explores the theme of racial tension and social change, set in the 1950s.
5. Do the Right Thing (1989) - As the provided dataset information notes, this film deals with racial tensions and social change in Bro

In [24]:
# additional question 2
question="What movies share the same genre as 12 Angry Men? Include some specific examples."
response = chain.invoke(question)
print(response)

Based on the dataset information, it seems that 12 Angry Men can be classified as a Drama, particularly a Courtroom Drama.

Movies that share a similar genre with 12 Angry Men include:

1. To Kill a Mockingbird (1962) - another classic Courtroom Drama based on Harper Lee's novel.
2. The Verdict (1982) - a Lawyer Drama where an embittered lawyer defends a client who is wrongly accused of murder.
3. A Time to Kill (1994) - a courtroom drama based on John Grisham's novel, which deals with issues of racial injustice and the loss of innocence.
4. Philadelphia (1993) - a courtroom drama starring Tom Hanks, which deals with themes of equality, justice, and personal struggle.

Please note that genre classification can be subjective, and there might be other movies that share a similar theme or tone with 12 Angry Men.

However, I do not have information about specific examples of movies that are known to share the same exact genre with 12 Angry Men.
